# ex03 · 过拟合诊断与评价指标（对应教材 4.4 + 一轮「评价指标」考点）

> **做题流程**：先预测（哪个阶数欠拟合/过拟合、指标手算），再运行验证。
> **做完再看** `solutions/ex03-答案.md`。
>
> 难度标记：🌱 基础（预测+验证）｜🔧 变式（改动观察）｜🚀 挑战（闭卷复现）
>
> 本节两件事：① 用多项式回归观察欠拟合/过拟合；② 手算 Accuracy/Precision/Recall/F1（一轮必考）。

In [ ]:
%matplotlib inline
import torch
import math
import numpy as np
from torch import nn
from torch.utils import data
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

## 第一部分 · 多项式回归观察欠拟合/过拟合

真实函数是一个 3 次多项式（加噪声），我们用不同阶数去拟合它。先读代码理解数据怎么造，再预测结果。

In [ ]:
max_degree = 20
n_train, n_test = 100, 100
true_w = np.zeros(max_degree)
true_w[0:4] = np.array([5, 1.2, -3.4, 5.6])   # 真实函数只有前 4 项（0~3 次）

features = np.random.normal(size=(n_train + n_test, 1))
np.random.shuffle(features)
poly_features = np.power(features, np.arange(max_degree).reshape(1, -1))
for i in range(max_degree):
    poly_features[:, i] /= math.gamma(i + 1)   # 除以阶乘，让不同阶特征同量级

labels = np.dot(poly_features, true_w)
labels += np.random.normal(scale=0.01, size=labels.shape)

true_w, features, poly_features, labels = [
    torch.tensor(x, dtype=torch.float32) for x in [true_w, features, poly_features, labels]]

def train(degree, num_epochs=400):
    """用前 degree 阶特征拟合，返回 (训练损失历史, 测试损失历史)"""
    loss = nn.MSELoss()
    net = nn.Sequential(nn.Linear(degree, 1, bias=False))  # 常数项已在特征里
    batch_size = 10
    train_iter = data.DataLoader(
        data.TensorDataset(poly_features[:n_train, :degree], labels[:n_train].reshape(-1, 1)),
        batch_size, shuffle=True)
    test_iter = data.DataLoader(
        data.TensorDataset(poly_features[n_train:, :degree], labels[n_train:].reshape(-1, 1)),
        batch_size, shuffle=False)
    trainer = torch.optim.SGD(net.parameters(), lr=0.01)
    train_ls, test_ls = [], []
    for epoch in range(num_epochs):
        for X, y in train_iter:
            l = loss(net(X), y)
            trainer.zero_grad()
            l.backward()
            trainer.step()
        if epoch == 0 or (epoch + 1) % 100 == 0:
            with torch.no_grad():
                tl = float(loss(net(poly_features[:n_train, :degree]), labels[:n_train].reshape(-1, 1)))
                te = float(loss(net(poly_features[n_train:, :degree]), labels[n_train:].reshape(-1, 1)))
            train_ls.append(tl)
            test_ls.append(te)
    print(f'degree={degree}: 训练损失 {train_ls[-1]:.4f}, 测试损失 {test_ls[-1]:.4f}')
    return train_ls, test_ls

### 题 1 🔧 预测：三个阶数的表现

先判断：degree = 2 / 4 / 20 分别对应 欠拟合 / 恰好 / 过拟合 中的哪个？各自训练损失和测试损失谁高谁低？

**【你的预测】**

In [ ]:
results = {}
for degree in [2, 4, 20]:
    results[degree] = train(degree, num_epochs=1500 if degree == 20 else 400)

In [ ]:
plt.figure(figsize=(10, 3))
for i, degree in enumerate([2, 4, 20]):
    tr, te = results[degree]
    plt.subplot(1, 3, i + 1)
    plt.plot(tr, label='train', marker='o')
    plt.plot(te, label='test', marker='o')
    plt.title(f'degree = {degree}')
    plt.xlabel('记录点'); plt.ylabel('loss')
    plt.legend()
plt.tight_layout()
plt.show()

## 第二部分 · 评价指标手算（面试高频）

一个二分类的混淆矩阵：TP=30、FP=10、FN=5、TN=55。先手算 Accuracy / Precision / Recall / F1，再运行对照。

**【你的预测】**（写出公式和手算过程）

In [ ]:
TP, FP, FN, TN = 30, 10, 5, 55
acc = (TP + TN) / (TP + FP + FN + TN)        # 准确率
prec = TP / (TP + FP)                        # 精确率
rec = TP / (TP + FN)                         # 召回率
f1 = 2 * prec * rec / (prec + rec)           # F1
print('Accuracy :', acc)
print('Precision:', round(prec, 4))
print('Recall   :', round(rec, 4))
print('F1-score :', round(f1, 4))

### 题 2 🔧 不均衡数据下 Accuracy 会骗人（面试高频）

先预测：100 个样本里 99 个负例、1 个正例，一个「永远猜负」的模型 Accuracy 是多少？它真的好吗？

In [ ]:
# 99 负例 + 1 正例；模型永远猜负
TP2, FP2, FN2, TN2 = 0, 0, 1, 99
acc2 = (TP2 + TN2) / (TP2 + FP2 + FN2 + TN2)
recall2 = TP2 / (TP2 + FN2)
print('永远猜负的 Accuracy:', acc2, '  Recall:', recall2)

## 小结与面试衔接

- 欠拟合：训练/测试损失**都高**；过拟合：训练低、测试高（差距大）——看 **train/test 的差距**
- 模型复杂度（阶数/隐藏单元数）越高越容易过拟合
- Accuracy 在不均衡数据下会「骗人」（99% 负例时全猜负也有 99% 准确率，但 Recall=0）
- 一轮「评价指标」考点：Accuracy/Precision/Recall/F1 的含义、不均衡时为什么 Accuracy 不可靠